# Instalar dependencias

In [1]:
%pip install --upgrade pip setuptools wheel

%pip install -U langchain-community

%pip install pypdf

%pip install --upgrade pip setuptools wheel

%pip install sentence_transformers==2.2.2 numpy==1.23.5

%pip check

  Using cached setuptools-78.1.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached wheel-0.45.1-py3-none-any.whl.metadata (2.3 kB)
Using cached setuptools-78.1.0-py3-none-any.whl (1.3 MB)
Using cached wheel-0.45.1-py3-none-any.whl (72 kB)
Note: you may need to restart the kernel to use updated packages.
  Using cached langchain_community-0.3.20-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_core-0.3.49-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain-0.3.21-py3-none-any.whl.metadata (7.8 kB)
  Using cached sqlalchemy-2.0.40-cp313-cp313-macosx_10_13_x86_64.whl.metadata (9.6 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached PyYAML-6.0.2-cp313-cp313-macosx_10_13_x86_64.whl.metadata (2.1 kB)
  Using cached aiohttp-3.11.14-cp313-cp313-macosx_10_13_x86_64.whl.metadata (7.7 kB)
  Using cached tenacity-9.0.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydanti

# Lectura PDF

In [2]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Cargar el PDF desde ,las carpetas de Drive
pdf_path = "./el principito.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Extraer el texto de cada página
text = "\n".join([doc.page_content for doc in documents])

# Chunks

In [12]:
# Dividir en fragmentos (chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 200)

chunks = text_splitter.split_text(text)  # revisar como usar .split_docuement para que sea compatible con chromaDB

# Transformar texto y vectores

In [21]:
from sentence_transformers import SentenceTransformer

def text_to_vector(text):

    # Cargar el modelo de embeddings
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # Convertir el texto en un vector numérico
    vector = model.encode(text)

    return vector

def vector_to_text(vector):
  # Cargar modelo
  model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

  # Convertir vector en texto
  text = model.decode(vector)

  return text

ModuleNotFoundError: No module named 'sentence_transformers'

# Chunks en vectores

In [6]:
chunk_vectors = []
chunk_vectors_len = []
for chunk in chunks:
  vector = text_to_vector(chunk)
  chunk_vectors.append(vector)
  chunk_vectors_len.append(len(vector))

print(len(chunk_vectors))
print(chunk_vectors_len)

264
[384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 

# Guardar Chunks

In [9]:
import numpy as np

# Convertir a un array de NumPy y guardar
np.save("chunks.npy", np.array(chunk_vectors))

# Leer desde el archivo .npy
loaded_vectors = np.load("chunks.npy")

print("Vectores cargados:", loaded_vectors)


ModuleNotFoundError: No module named 'numpy'

# Instalar SWIG

In [10]:
!apt-get install swig

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


zsh:1: command not found: apt-get


# Ball Tree

## Headers

In [52]:
%%file ball_tree.h

#ifndef BALL_TREE_H
#define BALL_TREE_H

#include <vector>

class BallTree {
private:
    struct Compare {
        int depth;
        Compare(int d) : depth(d) {}

        bool operator()(const std::vector<double>& a, const std::vector<double>& b) const {
            return a[depth % a.size()] < b[depth % a.size()];
        }
    };

    struct Node {
        std::vector<double> point;
        Node* left;
        Node* right;
    };

    Node* root;
    std::vector<std::vector<double> > data;

    // Función auxiliar para construir el árbol
    Node* build_tree(int left, int right, int depth);

    // Función auxiliar para encontrar el vecino más cercano
    void nearest_neighbor(Node* node, const std::vector<double>& target, Node*& best, double& best_dist, int depth);

public:
    BallTree(); // Constructor

    // Construye el árbol a partir de un conjunto de puntos
    void build(const std::vector<std::vector<double> >& points);

    // Encuentra el punto más cercano en el árbol
    std::vector<double> find_nearest(const std::vector<double>& target);
};

#endif // BALL_TREE_H


Overwriting ball_tree.h


## Class

In [53]:
%%file ball_tree.cpp
#include "ball_tree.h"
#include <cmath>
#include <limits>
#include <algorithm>

// Función para calcular la distancia euclidiana
double euclidean_distance(const std::vector<double>& a, const std::vector<double>& b) {
    double sum = 0.0;
    for (size_t i = 0; i < a.size(); ++i) {
        sum += (a[i] - b[i]) * (a[i] - b[i]);
    }
    return std::sqrt(sum);
}

// Constructor de la clase BallTree
BallTree::BallTree() : root(nullptr) {}

// Método para construir el árbol
void BallTree::build(const std::vector<std::vector<double> >& points) {
    this->data = points;
    this->root = build_tree(0, data.size() - 1, 0);
}

// Método privado para construir el árbol
BallTree::Node* BallTree::build_tree(int left, int right, int depth) {
    if (left > right) return nullptr;

    int mid = (left + right) / 2;
    std::nth_element(data.begin() + left, data.begin() + mid, data.begin() + right + 1, BallTree::Compare(depth));

    Node* node = new Node();
    node->point = data[mid];
    node->left = build_tree(left, mid - 1, depth + 1);
    node->right = build_tree(mid + 1, right, depth + 1);

    return node;
}

// Método privado para encontrar el vecino más cercano
void BallTree::nearest_neighbor(Node* node, const std::vector<double>& target, Node*& best, double& best_dist, int depth) {
    if (!node) return;

    double dist = euclidean_distance(target, node->point);
    if (dist < best_dist) {
        best_dist = dist;
        best = node;
    }

    int axis = depth % target.size();
    Node* next = target[axis] < node->point[axis] ? node->left : node->right;
    Node* other = (next == node->left) ? node->right : node->left;

    nearest_neighbor(next, target, best, best_dist, depth + 1);

    if (std::abs(target[axis] - node->point[axis]) < best_dist) {
        nearest_neighbor(other, target, best, best_dist, depth + 1);
    }
}

// Método público para encontrar el punto más cercano
std::vector<double> BallTree::find_nearest(const std::vector<double>& target) {
    Node* best = nullptr;
    double best_dist = std::numeric_limits<double>::max();
    nearest_neighbor(root, target, best, best_dist, 0);
    return best ? best->point : std::vector<double>();
}


Overwriting ball_tree.cpp


## Interfaz

In [54]:
%%file ball_tree.i
%module ball_tree
%{
#include "ball_tree.h"
%}

%include "std_vector.i"
%template(VectorDouble) std::vector<double>;
%template(VectorVectorDouble) std::vector<std::vector<double>>;

%include "ball_tree.h"

// Exponer la clase BallTree a Python
%feature("director") BallTree;


Overwriting ball_tree.i


# Ejecutar SWIG

In [2]:
!swig -c++ -python ball_tree.i
!g++ -fPIC -c ball_tree.cpp ball_tree_wrap.cxx -I$(python3-config --includes)
!g++ -shared ball_tree.o ball_tree_wrap.o -o _ball_tree.so \
-undefined dynamic_lookup $(python3-config --ldflags)


# Usar DLL

## Test del DLL

In [13]:
import ball_tree

# Crear un BallTree
tree = ball_tree.BallTree()

# Datos de ejemplo
points = [
    [2.0, 3.0],
    [5.0, 4.0],
    [9.0, 6.0],
    [4.0, 7.0],
    [8.0, 1.0],
    [7.0, 2.0]
]

# Construir el árbol
tree.build(points)

# Buscar el punto más cercano
query_point = [6.0, 3.0]
nearest = tree.find_nearest(query_point)
print("Nearest neighbor:", nearest)


Nearest neighbor: (5.0, 4.0)


## Uso en los chunks

In [23]:
import numpy as np
loaded_vectors = np.load('chunks.npy')
print(loaded_vectors[0])
treeChunks = ball_tree.BallTree()

import numpy as np

# Convertir de numpy.ndarray a lista de listas de float
loaded_vectors = loaded_vectors.astype(float).tolist()



# Ahora debería funcionar sin errores
treeChunks.build(loaded_vectors)

chunk = [ 1.51729314e-02 , 7.47226551e-02 , 3.04134395e-02 ,-2.52567083e-02,
  3.44625264e-02 , 4.07666527e-02  ,5.58534190e-02,  3.74462493e-02,
 -2.08935887e-02,  3.39054689e-02  ,1.20714299e-01, -4.31812443e-02,
 -6.88117072e-02 , 2.05941349e-02 , 6.02214783e-03,  3.30430754e-02,
 -6.73809424e-02 , 1.24954835e-01 , 4.94144559e-02,  6.74600750e-02,
  4.81393896e-02 ,-3.44640203e-02 ,-7.25436583e-02,  4.84762453e-02,
 -7.62665793e-02 , 1.87691096e-02 , 2.88231764e-02,  5.61301177e-03,
 -6.24075606e-02, -4.56007421e-02 ,-3.26343924e-02,  3.65714021e-02,
  9.23340470e-02, -9.98617057e-03  ,3.56453238e-04, -1.11581869e-02,
  2.47851685e-02 ,-2.59740446e-02 ,-5.84226921e-02,  4.92601693e-02,
 -1.67398714e-02, -1.97856855e-02 ,-2.69977609e-03, -4.82016280e-02,
 -8.86917487e-02, -1.00165963e-01 ,-4.55113649e-02,  4.20745090e-02,
  3.14197131e-02 ,-9.98437852e-02 ,-3.31681855e-02, -3.44412439e-02,
  1.07129812e-02 , 1.45469848e-02 ,-5.38676716e-02,  6.25108462e-03,
 -8.31466466e-02, -4.93312627e-02 , 4.48721163e-02,  4.43893820e-02,
 -3.27656232e-02,  5.37735485e-02, -5.13788573e-02,  3.82169932e-02,
  6.30703866e-02,  3.55488844e-02  ,3.04249991e-02, -6.65714294e-02,
 -1.31315157e-01,  5.49200699e-02 , 6.18499108e-02,  3.46569046e-02,
  4.45880322e-03, -3.58685963e-02, -4.52606268e-02 , 6.71660006e-02]
#treeChunks.build(loaded_vectors)

frase = "El principito"

# frase_vector = text_to_vector(frase)

nearest = treeChunks.find_nearest(chunk)

print(nearest)

# print()

[ 1.51729314e-02  7.47226551e-02  3.04134395e-02 -2.52567083e-02
  3.44625264e-02  4.07666527e-02  5.58534190e-02  3.74462493e-02
 -2.08935887e-02  3.39054689e-02  1.20714299e-01 -4.31812443e-02
 -6.88117072e-02  2.05941349e-02  6.02214783e-03  3.30430754e-02
 -6.73809424e-02  1.24954835e-01  4.94144559e-02  6.74600750e-02
  4.81393896e-02 -3.44640203e-02 -7.25436583e-02  4.84762453e-02
 -7.62665793e-02  1.87691096e-02  2.88231764e-02  5.61301177e-03
 -6.24075606e-02 -4.56007421e-02 -3.26343924e-02  3.65714021e-02
  9.23340470e-02 -9.98617057e-03  3.56453238e-04 -1.11581869e-02
  2.47851685e-02 -2.59740446e-02 -5.84226921e-02  4.92601693e-02
 -1.67398714e-02 -1.97856855e-02 -2.69977609e-03 -4.82016280e-02
 -8.86917487e-02 -1.00165963e-01 -4.55113649e-02  4.20745090e-02
  3.14197131e-02 -9.98437852e-02 -3.31681855e-02 -3.44412439e-02
  1.07129812e-02  1.45469848e-02 -5.38676716e-02  6.25108462e-03
 -8.31466466e-02 -4.93312627e-02  4.48721163e-02  4.43893820e-02
 -3.27656232e-02  5.37735